In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')



dane = pd.read_csv("Titanic Dataset.csv")

odsetek = dane.isna().mean() * 100
print(odsetek)

pclass        0.000000
survived      0.000000
name          0.000000
sex           0.000000
age          20.091673
sibsp         0.000000
parch         0.000000
ticket        0.000000
fare          0.076394
cabin        77.463713
embarked      0.152788
boat         62.872422
body         90.756303
home.dest    43.086325
dtype: float64


In [15]:
kolumny_z_brakami = ["age","fare","body"]
zmienne = ["sex","pclass","survived","embarked"]

flagi = dane[kolumny_z_brakami].isna()

for zmienna in zmienne:
    flagi[zmienna]=dane[zmienna]
    wynik = flagi.groupby(zmienna)[kolumny_z_brakami].mean() * 100
    
    print(wynik)
    


              age      fare       body
sex                                   
female  16.738197  0.000000  98.283262
male    21.945433  0.118624  86.595492
              age      fare       body
pclass                                
1       12.074303  0.000000  89.164087
2        5.776173  0.000000  88.808664
3       29.337094  0.141044  92.242595
                age      fare        body
survived                                 
0         23.485785  0.123609   85.043263
1         14.600000  0.000000  100.000000
                age      fare       body
embarked                                
C         21.481481  0.000000  90.740741
Q         59.349593  0.000000  94.308943
S         14.442013  0.109409  90.262582


Widzimy ze te osoby które przeżyły nie mają przypisanego numeru zwłok, pasażerowe z klasy trzeciej mają duzo mniej wpisanych informacji, podobnie jak ci z portu Q.

In [16]:
dane_baseline = dane.copy()
dane_advanced = dane.copy()

dane_baseline = dane_baseline.drop(['body','boat','cabin','home.dest','name','ticket'],axis=1)

num_cols = dane_baseline.select_dtypes(include=['float64', 'int64']).columns
for col in num_cols:
    dane_baseline[col]=dane_baseline[col].fillna(dane_baseline[col].mean())

cat_cols = dane_baseline.select_dtypes(include=['object']).columns
for col in cat_cols:
    dane_baseline[col]=dane_baseline[col].fillna(dane_baseline[col].mode()[0])


X_baseline = pd.get_dummies(dane_baseline.drop('survived',axis=1), drop_first=True)
y_baseline = dane_baseline['survived']



dane_advanced['cabin_missing'] = dane_advanced['cabin'].isna().astype(int)

dane_advanced['title'] = dane_advanced['name'].str.extract(' ([A-Za-z]+)\.', expand=False)

dane_advanced['age'] = dane_advanced.groupby('title')['age'].transform(lambda x: x.fillna(x.median()))
dane_advanced=dane_advanced.drop(['body','boat','cabin','home.dest','ticket','name'],axis=1)


num_cols = dane_advanced.select_dtypes(include=['float64', 'int64']).columns
for col in num_cols:
    dane_advanced[col]=dane_advanced[col].fillna(dane_advanced[col].mean())

cat_cols = dane_advanced.select_dtypes(include=['object']).columns
for col in cat_cols:
    dane_advanced[col]=dane_advanced[col].fillna(dane_advanced[col].mode()[0])


X_advanced = pd.get_dummies(dane_advanced.drop('survived',axis=1), drop_first=True)
y_advanced = dane_advanced['survived']


In [17]:
def eval_model(X,y,name):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f"Dokładność modelu ({name}): {acc:.4f}")

In [18]:
print("\n--- Porównanie Wyników ---")
eval_model(X_baseline,y_baseline, "Baselinowy")
eval_model(X_advanced,y_advanced, "Zaawansowany")


--- Porównanie Wyników ---
Dokładność modelu (Baselinowy): 0.7748
Dokładność modelu (Zaawansowany): 0.7863
